---
# <div align="center"><font color='blue'> </font></div>
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Week 7 Lab: Sequence Models & Text Classification</font></div>
---

This week we move from images to **text**. Unlike images, language depends heavily on word order, context, and memory. The sentence *"the dog bit the man"* means something very different from *"the man bit the dog"*. Same words, different order, completely different meaning.

This lab emphasises understanding **why** recurrent models are useful for text, not just how to use them.

## 🎯 Learning Objectives

By the end of this lab, you should be able to:

- Preprocess text data for deep learning (tokenisation, numericalisation, padding)
- Build a text preprocessing pipeline from scratch
- Build and train simple RNNs, LSTMs, and GRUs for text classification
- Understand the vanishing gradient problem in sequence models
- Use pretrained word embeddings (GloVe) as a form of transfer learning for NLP
- Compare sequence architectures systematically
- Perform error analysis on NLP models

**Prerequisites:** Weeks 3–6 (optimisation, CNNs, transfer learning, experimentation workflow)

**Duration:** ~2–3 hours

---


## Dataset: HappyDB

[HappyDB](https://github.com/megagonlabs/HappyDB) is a corpus of 100,000+ crowd-sourced happy moments. Each entry is a short text describing something that made a person happy, labelled with a **happiness category**:

| Category | Example | Count |
|---|---|---|
| affection | *"My wife surprised me with a hug when I came home"* | ~34k |
| achievement | *"I got a promotion at work today"* | ~34k |
| enjoy_the_moment | *"I watched a beautiful sunset"* | ~11k |
| bonding | *"I spent quality time with my friends"* | ~11k |
| leisure | *"I read a great book this weekend"* | ~7k |
| nature | *"I went for a walk in the park"* | ~2k |
| exercise | *"I ran 5 miles this morning"* | ~1k |

**Our task:** Given a happy moment text, predict its category.

> *Reference: Asai et al., "HappyDB: A Corpus of 100,000 Crowdsourced Happy Moments", LREC 2018.*

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y, lengths in loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x, lengths)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for x, y, lengths in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x, lengths)
            loss = criterion(outputs, y)
            total_loss += loss.item() * y.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


def run_experiment(model, optimizer, criterion, train_loader, val_loader,
                   epochs=10, name='Model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} — '
                  f'Train: {train_loss:.4f} / {train_acc:.4f} | '
                  f'Val: {val_loss:.4f} / {val_acc:.4f}')
    return history


def plot_experiments(results, show_train=True, show_val=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in results.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        if show_train:
            axes[0].plot(epochs, hist['train_loss'], label=f'{name} (Train)', marker='o', markersize=3)
            axes[1].plot(epochs, hist['train_acc'], label=f'{name} (Train)', marker='o', markersize=3)
        if show_val:
            axes[0].plot(epochs, hist['val_loss'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
            axes[1].plot(epochs, hist['val_acc'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
    for ax, ylabel in zip(axes, ['Loss', 'Accuracy']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(ylabel)
        ax.legend(fontsize=8); ax.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

---
# Part 1: Text Preprocessing Pipeline

Text data cannot be fed directly into a neural network — we need to convert words into numbers. The full pipeline is:

```
Raw text → Clean → Tokenise → Build vocabulary → Numericalise → Pad → Batch
```

This section walks through each step.

## Task 1.1 — Load and Explore the Dataset

In [ ]:
# Download HappyDB from GitHub
!wget -q https://raw.githubusercontent.com/megagonlabs/HappyDB/master/happydb/data/cleaned_hm.csv

df = pd.read_csv('cleaned_hm.csv')
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Use predicted_category as our label
# Drop rows with missing categories
df = df.dropna(subset=['predicted_category', 'cleaned_hm'])

# Show class distribution
print('Class distribution:')
print(df['predicted_category'].value_counts())
print(f'\nTotal usable samples: {len(df)}')

In [ ]:
# Visualise class distribution
fig, ax = plt.subplots(figsize=(10, 4))
counts = df['predicted_category'].value_counts()
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.set_title('HappyDB Category Distribution')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Show example sentences per category
for cat in df['predicted_category'].unique():
    examples = df[df['predicted_category'] == cat]['cleaned_hm'].head(2).tolist()
    print(f'\n{cat.upper()}:')
    for ex in examples:
        print(f'  "{ex[:100]}..."' if len(ex) > 100 else f'  "{ex}"')

In [ ]:
# Sentence length statistics
df['word_count'] = df['cleaned_hm'].apply(lambda x: len(str(x).split()))

print(f'Sentence length stats:')
print(df['word_count'].describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['word_count'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(df['word_count'].median(), color='red', linestyle='--', label=f'Median: {df["word_count"].median():.0f}')
ax.set_xlabel('Number of Words')
ax.set_ylabel('Count')
ax.set_title('Distribution of Sentence Lengths')
ax.legend()
plt.tight_layout()
plt.show()

### 🤔 Questions

1. Is the dataset balanced? Which classes are underrepresented?
2. Are sentence lengths consistent, or is there high variance?
3. Looking at the examples, do you think this classification task is easy or hard? Why?

*Double-click here to write your answers...*

## Task 1.2 — Tokenisation

Tokenisation splits text into individual units (tokens). We start with a simple approach and discuss its limitations.

In [ ]:
def tokenize(text):
    """Simple tokenizer: lowercase, remove punctuation, split on whitespace."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # remove punctuation
    return text.split()

# Test the tokenizer
test_sentences = [
    "I had a great dinner with my family!",
    "My son's first steps were amazing.",
    "I can't believe I won $100!",
]
for sent in test_sentences:
    print(f'  "{sent}"')
    print(f'  → {tokenize(sent)}\n')

### 🤔 Questions

1. What problems does this simple tokenizer have? (Hint: think about contractions like "can't", possessives like "son's", hyphenated words)
2. How might punctuation removal affect the model? When might punctuation carry meaning?
3. Would stemming or lemmatisation help for this task?

*Double-click here to write your answers...*

## Task 1.3 — Vocabulary Building

We need to map each unique word to an integer ID. Two special tokens are essential:
- **`<PAD>` (index 0):** Used to pad shorter sequences to equal length
- **`<UNK>` (index 1):** Replaces words not in the vocabulary (out-of-vocabulary words)

In [ ]:
# Tokenize all texts
df['tokens'] = df['cleaned_hm'].apply(tokenize)

# Count word frequencies
word_counts = Counter()
for tokens in df['tokens']:
    word_counts.update(tokens)

print(f'Total unique words: {len(word_counts)}')
print(f'\n20 most common words:')
for word, count in word_counts.most_common(20):
    print(f'  {word}: {count}')

In [ ]:
# Build vocabulary — keep only words that appear at least MIN_FREQ times
MIN_FREQ = 5
VOCAB_SIZE = 10000  # maximum vocabulary size

# Special tokens
PAD_IDX = 0
UNK_IDX = 1

# Build word-to-index mapping
word_to_idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
idx = 2

for word, count in word_counts.most_common(VOCAB_SIZE - 2):
    if count >= MIN_FREQ:
        word_to_idx[word] = idx
        idx += 1

idx_to_word = {v: k for k, v in word_to_idx.items()}
actual_vocab_size = len(word_to_idx)

print(f'Vocabulary size: {actual_vocab_size}')
print(f'Words excluded (freq < {MIN_FREQ}): {len(word_counts) - actual_vocab_size + 2}')

### 🤔 Questions

1. Why do we need the `<UNK>` token? What happens at test time when the model encounters a word it hasn't seen?
2. What happens if the vocabulary is too large? Too small?
3. Why did we use `MIN_FREQ=5`? What are the trade-offs?

*Double-click here to write your answers...*

## Task 1.4 — Numericalisation, Padding & DataLoader

Now we convert tokens to integer sequences, pad them to equal length, and wrap everything in a PyTorch Dataset and DataLoader.

In [ ]:
# Create label mapping
label_names = sorted(df['predicted_category'].unique())
label_to_idx = {label: i for i, label in enumerate(label_names)}
NUM_CLASSES = len(label_names)

print(f'Classes ({NUM_CLASSES}):')
for label, idx in label_to_idx.items():
    print(f'  {idx}: {label}')

In [ ]:
MAX_LEN = 50  # truncate sentences longer than this

class HappyDBDataset(Dataset):
    def __init__(self, texts, labels, word_to_idx, max_len=MAX_LEN):
        self.labels = labels
        self.max_len = max_len
        self.word_to_idx = word_to_idx

        # Convert tokens to indices
        self.sequences = []
        self.lengths = []
        for tokens in texts:
            indices = [word_to_idx.get(t, UNK_IDX) for t in tokens[:max_len]]
            self.lengths.append(len(indices))
            # Pad to max_len
            indices += [PAD_IDX] * (max_len - len(indices))
            self.sequences.append(indices)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.lengths[idx],
        )

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare data
all_tokens = df['tokens'].tolist()
all_labels = [label_to_idx[cat] for cat in df['predicted_category']]

# Subsample for faster iteration during the lab
# (increase to len(all_tokens) for full dataset when ready)
SUBSET_SIZE = 15000

if SUBSET_SIZE < len(all_tokens):
    all_tokens, _, all_labels, _ = train_test_split(
        all_tokens, all_labels, train_size=SUBSET_SIZE, random_state=42, stratify=all_labels
    )
    print(f'Using subset: {len(all_tokens)} samples')

# Train/val/test split: 70/15/15
X_temp, X_test, y_temp, y_test = train_test_split(
    all_tokens, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp  # 0.176 of 0.85 ≈ 0.15
)

train_dataset = HappyDBDataset(X_train, y_train, word_to_idx)
val_dataset = HappyDBDataset(X_val, y_val, word_to_idx)
test_dataset = HappyDBDataset(X_test, y_test, word_to_idx)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f'Training:   {len(train_dataset)}')
print(f'Validation: {len(val_dataset)}')
print(f'Test:       {len(test_dataset)} (DO NOT TOUCH until final evaluation)')

In [ ]:
# Verify: show a sample batch
sample_x, sample_y, sample_len = next(iter(train_loader))
print(f'Batch shape: {sample_x.shape}  (batch_size, max_len)')
print(f'Labels shape: {sample_y.shape}')

# Decode the first sample back to text
sample_tokens = [idx_to_word.get(idx.item(), '?') for idx in sample_x[0] if idx.item() != PAD_IDX]
print(f'\nFirst sample: "{" ".join(sample_tokens)}"')
print(f'Label: {label_names[sample_y[0].item()]}')
print(f'Length: {sample_len[0]}')

### 🤔 Questions

1. Why is padding necessary? What would happen if sequences in a batch had different lengths?
2. What problems arise with very long sequences? Why do we truncate at `MAX_LEN=50`?
3. Why do we return the original sequence length alongside the padded sequence?

*Double-click here to write your answers...*

---
# Part 2: Trivial Baseline Model — Bag of Embeddings

Before using sequence models, establish a simple baseline. The simplest approach is to **average the word embeddings** for all words in a sentence, then classify the resulting vector. This is sometimes called a "bag of embeddings" or "mean pooling" approach.

👉 **Key limitation:** This baseline ignores word order entirely. *"dog bites man"* and *"man bites dog"* produce the same representation.

In [ ]:
class BoEClassifier(nn.Module):
    """Bag of Embeddings: average word vectors, then classify."""
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x, lengths=None):
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        # Mean pooling (ignore padding)
        mask = (x != PAD_IDX).unsqueeze(-1).float()  # (batch, seq_len, 1)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(pooled)

In [ ]:
torch.manual_seed(0)
baseline = BoEClassifier(actual_vocab_size, embed_dim=64, num_classes=NUM_CLASSES).to(device)
optimizer = optim.Adam(baseline.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

all_results = {}
all_results['Baseline (BoE)'] = run_experiment(
    baseline, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='BoE'
)

plot_experiments(all_results)

### 🤔 Questions

1. What validation accuracy does the baseline achieve?
2. What information is lost by averaging word embeddings?
3. Can you think of examples where word order would change the category?

*Double-click here to write your answers...*

---
# Part 3: Simple RNN

## Why RNNs?

Unlike the bag-of-embeddings model, RNNs process text **sequentially**, one word at a time, and maintain a **hidden state** that acts as memory of what came before.

At each time step $t$, the RNN computes:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

The final hidden state $h_T$ summarises the entire sequence and is passed to a classifier.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, rnn_type='RNN'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        if rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.rnn_type = rnn_type
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths=None):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)

        # Get final hidden state
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]  # LSTM returns (hidden, cell)

        # hidden shape: (num_layers, batch, hidden_dim) → take last layer
        return self.fc(hidden[-1])

In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 128

torch.manual_seed(0)
rnn_model = RNNClassifier(
    actual_vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, rnn_type='RNN'
).to(device)

optimizer = optim.Adam(rnn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f'Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}')

all_results['Simple RNN'] = run_experiment(
    rnn_model, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='RNN'
)

plot_experiments(all_results)

**Think: Why Is the RNN Not Learning?**


The simple RNN barely improves over random guessing (~34% for the majority class), while the bag-of-embeddings baseline reached ~79%. Something is clearly wrong — but the architecture itself isn't the issue.

Look at how the RNN processes a padded sequence:
```
Sentence: "I had dinner with my family"  (6 words)
Padded:   [I, had, dinner, with, my, family, <PAD>, <PAD>, <PAD>, ... <PAD>]  (50 tokens)
                                              ↑                           ↑
                                         actual content ends        hidden state taken here
```
The RNN processes all 50 tokens sequentially, including the 44 padding tokens. By the time it reaches position 50, the hidden state has been overwritten by dozens of <PAD> embeddings. The meaningful information from the actual words has been washed away.


👉 The bag-of-embeddings model didn't have this problem — recall that it explicitly masked out padding tokens before computing the mean. The RNN has no such mechanism.
The Fix
Instead of taking the hidden state at the end of the padded sequence, we should take it at the actual last word — position 6 in the example above. This is why we stored the original sequence lengths in our Dataset.

The updated model (RNNClassifierV2) uses the lengths parameter to gather the output at the correct position for each sample in the batch.

🔑 Lesson: Padding is an implementation convenience, not real data. Always ensure your model ignores it — either through masking (as in the BoE baseline) or by indexing to the true sequence length (as in the fix below).

In [ ]:
class RNNClassifierV2(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, rnn_type='RNN'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        if rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.rnn_type = rnn_type
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths=None):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)  # output: (batch, seq_len, hidden_dim)

        if lengths is not None:
            # Gather the hidden state at the actual (non-padded) last position
            lengths_tensor = torch.tensor(lengths, dtype=torch.long).to(x.device)
            # Clamp to at least 1 to avoid index errors on empty sequences
            idx = (lengths_tensor - 1).clamp(min=0).unsqueeze(1).unsqueeze(2)
            idx = idx.expand(-1, 1, output.size(2))
            last_output = output.gather(1, idx).squeeze(1)  # (batch, hidden_dim)
        else:
            if self.rnn_type == 'LSTM':
                last_output = hidden[0][-1]
            else:
                last_output = hidden[-1]

        return self.fc(last_output)

In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 128

torch.manual_seed(0)
rnn_model = RNNClassifierV2(
    actual_vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, rnn_type='RNN'
).to(device)

optimizer = optim.Adam(rnn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f'Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}')

all_results['Simple RNN'] = run_experiment(
    rnn_model, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='RNN'
)

plot_experiments(all_results)

### 🤔 Questions

1. How does the RNN compare to the bag-of-embeddings baseline?
2. What sequence information can the RNN capture that the baseline cannot?
3. Look at the training curves — is the RNN stable during training?

*Double-click here to write your answers...*

---
# Part 4: The Vanishing Gradient Problem

Simple RNNs struggle with **long-term dependencies** because of the vanishing gradient problem. During backpropagation through time (BPTT), gradients are multiplied at each time step by the same weight matrix. If the largest eigenvalue of this matrix is < 1, the gradients shrink exponentially. After ~10–20 steps, the gradient becomes negligibly small.

This means simple RNNs effectively "forget" information from early in the sequence.

### Experiment: Short vs Long Sequences

Let's see this effect by comparing RNN performance on short vs long sequences.

In [ ]:
# Split data by sentence length
short_mask = df['word_count'] <= 15
long_mask = df['word_count'] > 30

print(f'Short sentences (≤15 words): {short_mask.sum()}')
print(f'Long sentences (>30 words):  {long_mask.sum()}')

# Train two models: one on short, one on long
# (For speed, just evaluate accuracy on each subset with the already-trained RNN)
rnn_model.eval()

def eval_subset(model, tokens_list, labels_list):
    ds = HappyDBDataset(tokens_list, labels_list, word_to_idx)
    loader = DataLoader(ds, batch_size=256, shuffle=False)
    _, acc = evaluate(model, loader, nn.CrossEntropyLoss())
    return acc

df_short = df[short_mask]
df_long = df[long_mask]

short_acc = eval_subset(rnn_model, df_short['tokens'].tolist(),
                        [label_to_idx[c] for c in df_short['predicted_category']])
long_acc = eval_subset(rnn_model, df_long['tokens'].tolist(),
                       [label_to_idx[c] for c in df_long['predicted_category']])

print(f'\nRNN accuracy on short sequences (≤15 words): {short_acc:.4f}')
print(f'RNN accuracy on long sequences  (>30 words):  {long_acc:.4f}')

### 🤔 Questions (Part 3)

1. Does performance degrade on longer sequences?
2. Why do simple RNNs struggle with long-term memory? Explain the vanishing gradient mechanism.
3. How might this affect real-world tasks like document classification or machine translation?

*Double-click here to write your answers...*

---
# Part 5: LSTM & GRU

**LSTMs** and **GRUs** are designed to solve the vanishing gradient problem using **gating mechanisms** that control what information to keep, forget, and output.

- **LSTM** (Long Short-Term Memory): Uses three gates (forget, input, output) and a separate cell state to maintain long-term memory.
- **GRU** (Gated Recurrent Unit): A simpler variant with two gates (reset, update). Fewer parameters, often similar performance.

In PyTorch, switching from RNN to LSTM or GRU requires just one line change — the `RNNClassifier` class above already supports all three via the `rnn_type` parameter.

In [ ]:
# Train LSTM
torch.manual_seed(0)
lstm_model = RNNClassifierV2(
    actual_vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, rnn_type='LSTM'
).to(device)
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

all_results['LSTM'] = run_experiment(
    lstm_model, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='LSTM'
)

# Train GRU
torch.manual_seed(0)
gru_model = RNNClassifierV2(
    actual_vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, rnn_type='GRU'
).to(device)
optimizer = optim.Adam(gru_model.parameters(), lr=1e-3)

all_results['GRU'] = run_experiment(
    gru_model, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='GRU'
)

plot_experiments(all_results)

### Architecture Comparison

In [ ]:
# Summary table
print(f'{"Model":<20} {"Best Val Acc":<15} {"Parameters":<15}')
print('=' * 50)
for name, model_obj in [('Baseline (BoE)', baseline), ('Simple RNN', rnn_model),
                         ('LSTM', lstm_model), ('GRU', gru_model)]:
    best_val = max(all_results[name]['val_acc'])
    n_params = sum(p.numel() for p in model_obj.parameters())
    print(f'{name:<20} {best_val:<15.4f} {n_params:<15,}')

In [ ]:
# Compare LSTM performance on short vs long sequences
lstm_short_acc = eval_subset(lstm_model, df_short['tokens'].tolist(),
                              [label_to_idx[c] for c in df_short['predicted_category']])
lstm_long_acc = eval_subset(lstm_model, df_long['tokens'].tolist(),
                             [label_to_idx[c] for c in df_long['predicted_category']])

print(f'Short sequences (≤15 words):')
print(f'  RNN:  {short_acc:.4f}')
print(f'  LSTM: {lstm_short_acc:.4f}')
print(f'\nLong sequences (>30 words):')
print(f'  RNN:  {long_acc:.4f}')
print(f'  LSTM: {lstm_long_acc:.4f}')

### 🤔 Questions

1. Which architecture performs best overall?
2. Which trains fastest per epoch?
3. Does the LSTM handle long sequences better than the simple RNN?
4. Why are gated models (LSTM/GRU) more stable during training?
5. The GRU has fewer parameters than the LSTM. Does this hurt or help on this dataset? Why?

*Double-click here to write your answers...*

---
# Part 6: Pretrained Word Embeddings

## Transfer Learning for NLP

Just as pretrained ImageNet features helped us in Week 5, **pretrained word embeddings** provide a head start for NLP. Instead of learning word representations from scratch, we use embeddings trained on massive text corpora.

**GloVe** (Global Vectors for Word Representation) embeddings were trained on 6 billion tokens from Wikipedia and web text. Each word is represented as a dense vector where similar words have similar vectors:
- "king" and "queen" are close together
- "happy" and "joyful" are close together
- "cat" and "democracy" are far apart

### Task 6.1 — Download and Load GloVe Embeddings

In [ ]:
# Download GloVe 6B (50d — smallest version, ~66MB)
import urllib.request
import zipfile

GLOVE_DIM = 50
GLOVE_FILE = f'glove.6B.{GLOVE_DIM}d.txt'

if not os.path.exists(GLOVE_FILE):
    print('Downloading GloVe embeddings...')
    urllib.request.urlretrieve(
        'https://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip'
    )
    with zipfile.ZipFile('glove.6B.zip', 'r') as z:
        z.extract(GLOVE_FILE)
    print('Done.')
else:
    print(f'{GLOVE_FILE} already exists.')

In [ ]:
# Load GloVe into a dictionary
def load_glove(filepath, dim):
    embeddings = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            if len(vector) == dim:
                embeddings[word] = vector
    return embeddings

glove = load_glove(GLOVE_FILE, GLOVE_DIM)
print(f'Loaded {len(glove):,} GloVe vectors ({GLOVE_DIM}d)')

# Check coverage
covered = sum(1 for w in word_to_idx if w in glove)
print(f'Vocabulary coverage: {covered}/{len(word_to_idx)} ({covered/len(word_to_idx)*100:.1f}%)')

### Task 6.2 — Create a Pretrained Embedding Matrix

In [ ]:
def create_embedding_matrix(word_to_idx, glove, embed_dim):
    """Create an embedding matrix with GloVe vectors for known words."""
    matrix = np.zeros((len(word_to_idx), embed_dim))
    found, missing = 0, 0

    for word, idx in word_to_idx.items():
        if word in glove:
            matrix[idx] = glove[word]
            found += 1
        else:
            # Random initialisation for unknown words
            matrix[idx] = np.random.normal(scale=0.6, size=embed_dim)
            missing += 1

    print(f'Embedding matrix: {matrix.shape}')
    print(f'  Found in GloVe: {found}, Missing: {missing}')
    return torch.tensor(matrix, dtype=torch.float32)

embedding_matrix = create_embedding_matrix(word_to_idx, glove, GLOVE_DIM)

### Task 6.3 — LSTM with Pretrained Embeddings

In [ ]:
class PretrainedLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_classes, freeze_embeddings=True):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embeddings, padding_idx=PAD_IDX
        )
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, lengths=None):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)  # output: (batch, seq_len, hidden_dim)

        if lengths is not None:
            lengths_tensor = torch.tensor(lengths, dtype=torch.long).to(x.device)
            idx = (lengths_tensor - 1).clamp(min=0).unsqueeze(1).unsqueeze(2)
            idx = idx.expand(-1, 1, output.size(2))
            last_output = output.gather(1, idx).squeeze(1)
        else:
            last_output = output[:, -1, :]  # fallback: last position

        return self.fc(last_output)

In [ ]:
# Compare: frozen vs trainable pretrained embeddings
embed_results = {}

for freeze, name in [(True, 'GloVe (frozen)'), (False, 'GloVe (fine-tuned)')]:
    torch.manual_seed(0)
    model = PretrainedLSTM(
        embedding_matrix, HIDDEN_DIM, NUM_CLASSES, freeze_embeddings=freeze
    ).to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    embed_results[name] = run_experiment(
        model, optimizer, criterion, train_loader, val_loader,
        epochs=10, name=name
    )
    print()

# Add the learned-from-scratch LSTM for comparison
embed_results['Random init'] = all_results['LSTM']

plot_experiments(embed_results)

### 🤔 Questions

1. Do pretrained embeddings improve performance compared to random initialisation?
2. Does freezing the embeddings help or hurt? Why?
3. Why do pretrained embeddings help even though they were trained on a different task (language modelling, not classification)?
4. When might pretrained embeddings fail? (Hint: think about domain-specific vocabulary like medical terms or slang)

*Double-click here to write your answers...*

---
# Part 7: Error Analysis

Following the Week 6 workflow, let's understand what the model gets wrong.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Use the best model so far
best_model_name = max(embed_results, key=lambda k: max(embed_results[k]['val_acc']))
print(f'Best model: {best_model_name}')

# Retrain the best for evaluation
# (use the model object that's still in memory)
# Get predictions on validation set
all_preds, all_labels_list, all_texts = [], [], []

model.eval()
with torch.inference_mode():
    for x, y, lengths in val_loader:
        x = x.to(device)
        outputs = model(x, lengths)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels_list.extend(y.numpy())

all_preds = np.array(all_preds)
all_labels_arr = np.array(all_labels_list)

print(classification_report(all_labels_arr, all_preds, target_names=label_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels_arr, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

### Inspect Misclassified Examples

In [ ]:
# Show misclassified examples
errors = np.where(all_preds != all_labels_arr)[0]
print(f'Misclassified: {len(errors)} / {len(all_labels_arr)} ({len(errors)/len(all_labels_arr)*100:.1f}%)\n')

# Show some misclassified examples
val_texts = X_val  # the tokenised validation texts

print('Sample misclassified examples:')
print('-' * 80)
for i in range(min(10, len(errors))):
    idx = errors[i]
    text = ' '.join(val_texts[idx])
    true_label = label_names[all_labels_arr[idx]]
    pred_label = label_names[all_preds[idx]]
    print(f'  True: {true_label:<20} Pred: {pred_label:<20}')
    print(f'  "{text[:120]}..."' if len(text) > 120 else f'  "{text}"')
    print()

### Analyse: Word Order & Negation

NLP models can struggle with negation, sarcasm, and subtle contextual shifts. Let's test a few examples:

In [ ]:
# Test the model on crafted examples
test_sentences = [
    "I had dinner with my family",
    "I went to the gym and ran five miles",
    "I got a promotion at work today",
    "I enjoyed a beautiful sunset at the beach",
    "I played with my dog in the park",
    "My friends came over for a game night",
]

model.eval()
for sent in test_sentences:
    tokens = tokenize(sent)
    indices = [word_to_idx.get(t, UNK_IDX) for t in tokens[:MAX_LEN]]
    indices += [PAD_IDX] * (MAX_LEN - len(indices))
    x = torch.tensor([indices], dtype=torch.long).to(device)

    with torch.inference_mode():
        output = model(x, [len(tokens)])
        probs = torch.softmax(output, dim=1)
        pred_idx = output.argmax(dim=1).item()

    print(f'  "{sent}"')
    print(f'  → {label_names[pred_idx]} ({probs[0][pred_idx].item():.2%} confidence)\n')

### 🤔 Questions

1. What types of mistakes occur most often? Which categories are most confused?
2. Are there examples where the model's prediction seems reasonable even if it doesn't match the label?
3. Do the crafted examples produce sensible predictions? Any surprises?
4. How would sarcasm or negation affect this model?

*Double-click here to write your answers...*

---
# Part 8: Mini Challenge

Improve the text classifier as much as possible. Try one or more of:

- **Handle Imbalance:** Stratified sampling in the DataLoader, Weighted loss function
- **Bidirectional LSTM/GRU:** Process the sequence in both directions
- **Larger GloVe embeddings:** Use 100d or 200d instead of 50d
- **Multi-layer RNN:** `nn.LSTM(embed_dim, hidden_dim, num_layers=2)`
- **Better tokenisation:** Handle contractions, punctuation, etc.
- **Learning rate tuning:** Try different LR values
- **Longer training:** More epochs with a scheduler

### Final Test Evaluation

In [ ]:
# 🚨 Do this ONCE with your best model
# best_model.eval()
# test_loss, test_acc = evaluate(best_model, test_loader, criterion)
# print(f'Final Test Accuracy: {test_acc:.4f}')

---
# Part 8: Reflection

1. Why are sequence models better suited to text than MLPs or bag-of-words models?
2. What was the biggest limitation of simple RNNs, and how do LSTMs/GRUs address it?
3. When would you choose GRU vs LSTM?
4. What role do pretrained embeddings play in NLP? How is this analogous to pretrained CNNs in computer vision?
5. What was the most surprising thing you learned in this lab?

*Double-click here to write your reflection...*



## 🔑 Key Takeaway

Sequence models process information over time and can capture contextual relationships in language. The progression from bag-of-words → RNN → LSTM/GRU mirrors the history of NLP: each step addresses a specific limitation of the previous approach. Pretrained word embeddings are the NLP equivalent of pretrained CNN features — they provide a head start by encoding linguistic knowledge learned from massive text corpora.